In [ ]:
import re, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import nltk
import tensorflow as tf
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

nltk.download("stopwords", quiet=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("Setup complete")

In [ ]:
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df = pd.read_csv(url, sep="\t", header=None, names=["label", "message"])

print("Shape:", df.shape)
df.head()

In [ ]:
print("Missing values:\n", df.isnull().sum(), "\n")
print("Duplicate messages:", df.duplicated().sum(), "\n")
print("Class counts:\n", df["label"].value_counts(), "\n")
print("Class share (%):\n", (df["label"].value_counts(normalize=True) * 100).round(2), "\n")

df["length"] = df["message"].str.len()
df.groupby("label")["length"].describe().round(1)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.countplot(data=df, x="label", hue="label", palette=["#48bb78", "#f56565"], legend=False, ax=ax[0])
ax[0].set_title("Ham vs Spam messages"); ax[0].set_xlabel("Label"); ax[0].set_ylabel("Count")
sns.histplot(data=df, x="length", hue="label", bins=40, palette=["#48bb78", "#f56565"], ax=ax[1])
ax[1].set_title("Message length by class"); ax[1].set_xlabel("Characters")
plt.tight_layout(); plt.show()

In [ ]:
ps = PorterStemmer()
stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = text.lower()
    text = "".join([c for c in text if c.isalpha() or c == " "])
    words = text.split()
    words = [ps.stem(w) for w in words if w not in stop_words]
    return " ".join(words)

df["clean"] = df["message"].apply(clean_text)

for i in [0, 2, 5]:
    print("ORIGINAL:", df["message"][i])
    print("CLEANED :", df["clean"][i])
    print()

In [ ]:
y = (df["label"] == "spam").astype(int).values

X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["clean"], y, test_size=0.2, random_state=SEED, stratify=y
)

tfidf = TfidfVectorizer(max_features=3000)
X_train = tfidf.fit_transform(X_train_text).toarray()
X_test = tfidf.transform(X_test_text).toarray()

print("Train matrix:", X_train.shape)
print("Test matrix :", X_test.shape)
print("Spam share in train: %.2f%%" % (y_train.mean() * 100))
print("Spam share in test : %.2f%%" % (y_test.mean() * 100))

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(3000,)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=10, batch_size=32,
    validation_split=0.1,
    verbose=2,
)

In [ ]:
y_prob = model.predict(X_test, verbose=0).ravel()
y_pred = (y_prob > 0.5).astype(int)

print("Accuracy : %.4f" % accuracy_score(y_test, y_pred))
print("Precision: %.4f" % precision_score(y_test, y_pred))
print("Recall   : %.4f" % recall_score(y_test, y_pred))
print("F1-score : %.4f" % f1_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred, target_names=["ham", "spam"], digits=4))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print(cm)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["ham", "spam"], yticklabels=["ham", "spam"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion matrix"); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(history.history["accuracy"], label="train"); ax[0].plot(history.history["val_accuracy"], label="validation")
ax[0].set_title("Accuracy"); ax[0].set_xlabel("Epoch"); ax[0].legend()
ax[1].plot(history.history["loss"], label="train"); ax[1].plot(history.history["val_loss"], label="validation")
ax[1].set_title("Loss"); ax[1].set_xlabel("Epoch"); ax[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
model.save("spam_classifier.keras")
joblib.dump(tfidf, "tfidf_vectorizer.pkl")

import os
print("spam_classifier.keras :", os.path.getsize("spam_classifier.keras") // 1024, "KB")
print("tfidf_vectorizer.pkl  :", os.path.getsize("tfidf_vectorizer.pkl") // 1024, "KB")

In [ ]:
def predict_spam(text):
    vec = tfidf.transform([clean_text(text)]).toarray()
    return float(model.predict(vec, verbose=0)[0][0])

examples = [
    "Hey, are we still meeting for lunch today? Let me know!",
    "CONGRATULATIONS! You've been selected to win a $1000 Walmart Gift Card. Click here to claim!",
    "Your package is waiting for delivery. Please confirm your address at: http://bit.ly/malicious-link",
]
for msg in examples:
    p = predict_spam(msg)
    label = "SPAM" if p > 0.5 else "HAM"
    conf = p * 100 if p > 0.5 else (1 - p) * 100
    print(f"{label:4s} ({conf:.2f}%)  |  {msg}")

In [ ]:
my_message = "Free entry! Text WIN to 80082 now to claim your prize"
p = predict_spam(my_message)
print("SPAM" if p > 0.5 else "HAM", "- spam probability: %.2f%%" % (p * 100))

In [ ]:
print("Spam checker ready. Type a message and press Enter (type 'quit' to stop).\n")

while True:
    msg = input("Enter message: ").strip()
    if msg.lower() in ("quit", "exit", "q"):
        print("Stopped.")
        break
    if msg == "":
        print("Please enter a message.\n")
        continue
    p = predict_spam(msg)
    if p > 0.5:
        print(f"SPAM  - probability {p * 100:.2f}%\n")
    else:
        print(f"HAM (legitimate) - probability {(1 - p) * 100:.2f}%\n")

Spam checker ready. Type a message and press Enter (type 'quit' to stop).

Enter message: "Free entry! Text 22222 now to claim the prize"
SPAM  - probability 100.00%

Enter message: hello
HAM (legitimate) - probability 96.47%

Enter message: hi 
HAM (legitimate) - probability 99.38%

Enter message: how are you
HAM (legitimate) - probability 99.48%

